# MiniMe — Kaggle launcher

A launcher, not a place for logic: it checks out a pinned commit, checks the GPUs and the
data, and hands a list of jobs to `minime.training.launch.run_queue`, which runs
`scripts/train.py` once per job, one job per GPU. Everything that decides a result lives in
the repository, under test.

**Once**

1. Kaggle → *Settings* → verify a phone number (Internet access needs it).
2. GitHub → *Settings → Developer settings → Fine-grained tokens*: a token with
   **read-only Contents** on `knam2609/minime_llm` and nothing else. Here: *Add-ons →
   Secrets* → add it as `GITHUB_TOKEN` and attach it to this notebook.
3. On the laptop: `python scripts/checksum_shards.py $MINIME_DATA_ROOT/shards/general`, then
   upload that directory, `SHA256SUMS` included, as a **private** dataset. Training needs no
   tokenizer file: the tokenizer never leaves the laptop.
4. For jobs that mix in the owner's chat (P4; M10's sweep, its one extension and the
   flagship sessions) — and only with the owner's yes for those jobs —
   `python scripts/stage_personal_upload.py`, then upload
   `$MINIME_DATA_ROOT/upload/personal-chat-train` as a second **private** dataset. It holds
   `chat.train` alone; the personal val and test splits stay on the laptop. Tokenised text
   is recoverable text: delete the dataset after the last run that reads it, and every
   chat-trained checkpoint in a version's output once its download is verified (below).

**Each session**

1. *Settings*: Accelerator **GPU T4 x2**, Internet **on**. *Add Input*: each dataset in
   `DATASETS`, and no other — plus, with `RESUME = True`, the earlier session's output.
2. Edit the parameters cell — `COMMIT`, `JOBS`, `DATASETS`, `EXPERIMENT`, `RESUME`,
   `SESSION_HOURS`, `SWEEP_EXTENSION` — and nothing else.
3. *Save Version → Save & Run All (Commit)*. It runs headless (Kaggle's limit is 12 h).
   Every job pauses cleanly `SESSION_HOURS` after the session started (exit 75, `last.pt`
   kept, the manifest marked paused); the results cell still runs.
4. From the version's *Output*, download `results.tar.gz`; unpack it **in a scratch
   directory**, never the repo root, and copy the new `experiments/<run>/` directories in.
   Append `kaggle-costs.csv`'s rows to `experiments/costs.csv` (`tail -n +2`). The
   checkpoints stay in the version's output under `experiments/<run>/`; download the ones
   needed, then check each against the session's own hashes before the cloud copy is
   deleted:
   `python scripts/verify_download.py <file> --run-dir <scratch>/experiments/<run>`
   (`--name last.pt` when the browser saved it as `last.zip`).

**A run longer than a session** (the flagship): the next session attaches the earlier
session's output as an input, keeps the same `JOBS` and sets `RESUME = True`. Each job's
paused run is found there and resumed only if it is the same job, at the same commit, on
the same GPU type, with the files that session hashed. Keep the earlier output until the
resumed session's checkpoints are home and verified.

A job that fails does not stop the others; the last cell lists each job's outcome, and
`scripts/pick_lr.py` / `pick_m10_lr.py` refuse a sweep with an unfinished run.

In [ ]:
# ---- parameters: the only cell to edit ------------------------------------
REPO = "knam2609/minime_llm"
# The code every job runs; it must be on GitHub.  Bump it, in a commit of its
# own, whenever the jobs need a newer config or fix.
COMMIT = "53ee177b1093f4b22ee1f3ac26faaac08ffafd0a"
# (config relative to the repository root, ["--set" overrides]); one run each.
JOBS = [
    # M10's flagship: the deliverable model, the owner's chat from step 0
    # (0.116%), AdamW at the sweep's pick 8e-4 (reports/m10/sweep.json).
    # ~9.3 h projected: if the session's deadline comes first it pauses, and
    # the next session runs these same JOBS with RESUME = True.  Reads the
    # personal dataset: see DATASETS.
    ("configs/flagship.yaml", []),
]
# The shard datasets to attach, by manifest kind; the first holds the val set.
# "personal" (the staged chat.train) only for a session whose jobs need it.
DATASETS = ["general", "personal"]
EXPERIMENT = "flagship"                       # the costs.csv label
# True: continue each job's paused run from the earlier session's output (attached).
RESUME = False
# Every job pauses cleanly this long after the session starts (Kaggle stops at 12 h).
SESSION_HOURS = 11.0
# M10's sweep only: after the four points, queue the edge rule's one extension.
SWEEP_EXTENSION = False
VENUE = "kaggle-free"
GPU = "Tesla T4"                              # one GPU type for every job in a sweep
TOKENIZER_ID = "minime-16k@92bb9c71bf6e2da5"  # the frozen tokenizer, never another

In [ ]:
import subprocess
import sys
from datetime import datetime, timedelta, timezone

SESSION_START = datetime.now(timezone.utc)
STOP_BY = SESSION_START + timedelta(hours=SESSION_HOURS)
assert 0 < SESSION_HOURS < 12, "Kaggle stops a session at 12 h: pause before it"
assert sys.version_info >= (3, 11), f"Python {sys.version}: the code needs 3.11+"
# nvidia-smi, not torch: this kernel must not hold a CUDA context on a GPU a job needs.
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True, check=True).stdout
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError("no GPU: Settings -> Accelerator -> GPU T4 x2") from None
names = [line.strip() for line in smi.splitlines() if line.strip()]
assert names and all(n == GPU for n in names), f"GPUs {names}, expected {GPU}"
N_GPUS = len(names)
print(f"Python {sys.version.split()[0]} | {N_GPUS} x {GPU} | jobs pause after {STOP_BY:%Y-%m-%d %H:%M} UTC")

In [ ]:
import base64
from pathlib import Path

from kaggle_secrets import UserSecretsClient

CODE = Path("/tmp/minime_llm")  # not /kaggle/working: the code is not an output
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"{REPO.split('/')[0]}:{token}".encode()).decode()


def git(*args, cwd=CODE):
    # The token rides in a per-command header: never in .git/config, never printed.
    cmd = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}", *args]
    out = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if out.returncode:
        err = out.stderr.replace(token, "***").replace(auth, "***")
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    return out.stdout.strip()


if CODE.exists():
    git("fetch", "--quiet", "origin")
else:
    git("clone", "--quiet", f"https://github.com/{REPO}.git", str(CODE), cwd=None)
git("checkout", "--quiet", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT, "COMMIT must be a full 40-character sha"
# A dirty tree would record every manifest's git revision as "<sha>-dirty".
assert git("status", "--porcelain") == "", "the checkout is not clean"
print(git("log", "-1", "--format=%H %s"))


In [ ]:
sys.path.insert(0, str(CODE / "src"))
from minime.training.launch import Job, find_shard_dir, run_queue, verify_checksums, write_run_checksums

DATA = [find_shard_dir("/kaggle/input", kind, TOKENIZER_ID) for kind in DATASETS]
for data_dir in DATA:
    print(f"{data_dir}: {verify_checksums(data_dir)} files match SHA256SUMS")

In [ ]:
WORK = Path("/kaggle/working")  # run directories, logs, costs.csv: the notebook's output
QUEUE = dict(repo=CODE, data=DATA, venue=VENUE, experiment=EXPERIMENT, gpus=N_GPUS, work_dir=WORK,
             resume_root=Path("/kaggle/input") if RESUME else None, stop_by=STOP_BY, accelerator=GPU)
results = run_queue([Job(config, tuple(overrides), resume=RESUME) for config, overrides in JOBS], **QUEUE)

if SWEEP_EXTENSION:
    from minime.training.m10 import next_sweep_job

    extension, why = next_sweep_job([WORK / "experiments" / r.run_name for r in results])
    print(why)
    if extension:
        results += run_queue([Job(extension)], **QUEUE)

In [ ]:
import shutil

# Each run directory's files hashed beside them, for checking the downloads against.
for sums in write_run_checksums(WORK):
    print(f"hashed {sums.parent.name}")
# For the repository: each run directory without its checkpoints, the logs, and this
# session's cost rows -- renamed, so unpacking never replaces the repo's costs.csv.
costs = WORK / "experiments" / "costs.csv"
if costs.exists():
    shutil.copyfile(costs, WORK / "kaggle-costs.csv")
packed = [p for p in ("experiments", "logs", "kaggle-costs.csv") if (WORK / p).exists()]
subprocess.run(["tar", "--exclude=*.pt", "--exclude=experiments/costs.csv",
                "-czf", str(WORK / "results.tar.gz"), "-C", str(WORK), *packed], check=True)
for r in results:
    outcome = "ok    " if r.ok else "PAUSED" if r.paused else "FAILED"
    print(f"{outcome}  {r.run_name:32s} exit {r.returncode:3d}  {r.seconds / 60:6.1f} min  gpu {r.gpu}")